# 01 - Data Cleaning

Load, clean, and preprocess resume text data for downstream ML tasks.

In [ ]:
# Install dependencies if needed
!pip install -q pandas numpy scikit-learn faker matplotlib seaborn spacy
!python -m spacy download en_core_web_sm

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import spacy
from collections import Counter

print("Libraries loaded successfully")

In [ ]:
# Load the candidates dataset
df = pd.read_csv('../datasets/candidates.csv')
print(f"Loaded {len(df)} candidates")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Basic statistics
print(f"Shape: {df.shape}")
print(f"\nExperience range: {df['experience'].min()} - {df['experience'].max()} years")
print(f"Avg experience: {df['experience'].mean():.1f} years")
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^a-zA-Z0-9.,;:!?\-\s]', '', text)
    return text.strip().lower()

df['clean_text'] = df['resume_text'].apply(clean_text)
print("Text cleaned successfully")
print(f"Sample: {df['clean_text'].iloc[0][:100]}...")

In [ ]:
# Load spaCy for lemmatization
nlp = spacy.load('en_core_web_sm')

def lemmatize_text(text):
    doc = nlp(text[:500])  # Limit for performance
    return ' '.join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct])

# Process a sample
sample = df['clean_text'].iloc[0]
lemmatized = lemmatize_text(sample)
print(f"Original: {sample[:80]}...")
print(f"Lemmatized: {lemmatized[:80]}...")

In [ ]:
# Extract skill counts
all_skills = []
for skills in df['skills']:
    all_skills.extend([s.strip() for s in skills.split(',')])

skill_counts = Counter(all_skills)
top_skills = skill_counts.most_common(15)

print("Top 15 skills across all candidates:")
for skill, count in top_skills:
    print(f"  {skill}: {count}")

In [ ]:
# Save processed data
processed_df = df[['id', 'name', 'email', 'skills', 'experience', 'education', 'clean_text']].copy()
processed_df.to_csv('../datasets/processed_candidates.csv', index=False)
print(f"Saved processed dataset: {len(processed_df)} rows")
print("Data cleaning complete!")